# TM-RugPull dataset initial analysis

## Data collection

In [ ]:
#Loading data from .xlsx file

import pandas as pd
import numpy as np
import matplotlib.pyplot as pyplot

file = 'data/TM-RugPull.xlsx'

data = pd.read_excel(file)

#Remove a space in the end of some column names
data.columns = data.columns.str.strip()

print(data.shape)

pd.set_option('display.max_columns', None)

data.head(5)

In [10]:
#Check the balance between classes in the whole dataset
#TODO: display in %

class_counts = data.groupby('class').size()
print(class_counts)

class
normal    401
scam      599
dtype: int64


##### Create a test set and a validation test

In [ ]:
#Create a test set and a validation set from the raw data to avoid data leakage
#Validation set size is about 10,5% and test set size is about 24,5% from the whole data set
#TODO reference to AML tutorial

from sklearn.model_selection import train_test_split

#define size of data both for test and validatioin sets, define the seed for all subsequent experiments
test_and_val_size = 0.35
seed = 7

#Split the data first on train set and set for test and validation
train_set, test_and_val_set = train_test_split(data, test_size=test_and_val_size, random_state=seed, stratify=data['class'])

#Split the part for test and validation into test set and validation set
test_set, val_set = train_test_split(test_and_val_set, test_size=0.3, random_state=seed, stratify=test_and_val_set['class'])

#Output the shapes to verify the splits
print("Training set shape:", train_set.shape)
print("Test set shape:", test_set.shape)
print("Validation set shape:", val_set.shape)

#Create a list of sets to perform further feature engeneering on all subsets of data
data_sets = [train_set, test_set, val_set]

In [ ]:
# Verify that there is no overlap between sets, no data leak at this stage
print("Overlap between train and test:", np.intersect1d(train_set.index, test_set.index).size)
print("Overlap between train and validation:", np.intersect1d(train_set.index, val_set.index).size)
print("Overlap between test and validation:", np.intersect1d(test_set.index, val_set.index).size)

## Data analysis

In [ ]:
#Check general info about data

print("\nDataset information:")
data.info()

In [ ]:
#Deleting columns that are not useful for further analysis

for set in data_sets:
    set.drop(columns=['Project Title', 'Sign', 'website', 'x profile', 'Smart Contract (online)', 'smart Contract (offline)', 'project starting date', 'project end date'], inplace=True)

In [ ]:
train_set.head(10)

In [8]:
#Check the description of data

description = train_set.describe()
description

,MaxPrice (Quarter 1),MaxPrice (Quarter 2),MaxPrice (Quarter 3),MaxPrice (Quarter 4),Total Variance,Variance of holders with more than 1% tokens,Google results for project title (first day),Google results for project title (project duration/2),Google results for project x profile (first days),Google results for project x profile (duration/2)
count,6.500000e+02,6.500000e+02,6.500000e+02,6.500000e+02,6.500000e+02,6.500000e+02,6.500000e+02,650.000000,650.000000,650.000000
mean,2.912336e+11,2.907117e+11,2.907107e+11,2.908765e+11,2.652559e+67,1.137186e+40,3.407672e+04,2156.347692,2147.495385,722.503077
std,7.411538e+12,7.411546e+12,7.411547e+12,7.411541e+12,6.762625e+68,2.883871e+41,6.593665e+05,17810.171807,14184.099535,4710.665689
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,8.239937e-18,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
25%,1.292500e-03,7.529400e-04,5.987500e-04,5.155050e-04,1.315515e+11,1.584406e+13,3.000000e+00,2.000000,4.000000,2.000000
50%,1.462085e-01,7.300000e-02,5.181000e-02,4.682000e-02,1.802270e+13,4.366251e+15,5.000000e+01,9.000000,47.000000,18.000000
75%,2.374000e+00,1.309500e+00,1.004000e+00,1.007450e+00,6.704701e+16,3.606957e+18,3.545000e+02,197.500000,325.000000,122.000000
max,1.889581e+14,1.889581e+14,1.889581e+14,1.889581e+14,1.724138e+70,7.352468e+42,1.640000e+07,343000.000000,240000.000000,94000.000000


In [9]:
#Check the balance between classes in the training set

class_counts = train_set.groupby('class').size()
print(class_counts)

class
normal    261
scam      389
dtype: int64


In [11]:
#Replace strings in 'Blockchain', 'Blockchain Type', 'class' columns with numbers
#to make all features numeric for further analysis

#TODO: should I enforce specific values to each type of blockchain and its type or rely on LabelEncoder embedded?
#TODO: reference from notes

from sklearn.preprocessing import LabelEncoder

for set in data_sets:
    le = LabelEncoder()
    set['Blockchain'] = le.fit_transform(set['Blockchain'])
    set['Blockchain Type'] = le.fit_transform(set['Blockchain Type'])
    set['class'] = set['class'].map({'normal': 0, 'scam': 1})

train_set

,MaxPrice (Quarter 1),MaxPrice (Quarter 2),MaxPrice (Quarter 3),MaxPrice (Quarter 4),Blockchain,the number of Transactions,Token concentration ratio per holder,Total Variance,Variance of holders with more than 1% tokens,Token balance,first deposits,Blockchain Type,class,Google results for project title (first day),Google results for project title (project duration/2),Google results for project website (first day),Google results for project website (duration/2),Google results for project x profile (first days),Google results for project x profile (duration/2)
380,5.334000e-02,5.334000e-02,5.334000e-02,1.758000e-01,4,41,30,5.733989e+08,2.112925e+09,160000,0.05334,3,1,0,0,0,0,0,0
366,4.148000e-06,4.074000e-06,4.074000e-06,4.074000e-06,4,74,62,1.049362e+28,1.359328e+29,1000066666666660,0.000004,3,1,1,1,0,0,0,0
967,1.118000e+01,5.190000e+00,3.140000e+00,1.800000e+01,2,90329,2089,6.759598e+07,1.039946e+10,850000,11.18,5,0,5,3,2,2,140,66
185,5.199000e-05,0.000000e+00,3.385000e-05,4.763000e-05,2,1383,1234,4.183023e+26,7.124242e+28,1000000000000000,0.000052,5,1,2,0,1,0,3,2
400,2.282000e-03,8.994000e-05,8.424000e-05,6.630000e-05,7,2592,736,1.245419e+07,4.681463e+08,195379.0521,0.002282,3,1,867,672,0,0,5,31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
931,8.400000e-02,3.600000e-02,2.100000e-02,1.400000e-02,4,151072,3434,1.513178e+15,2.276288e+17,6450034952.013,0.053,3,0,79,7,3050,245,36,8
314,3.634000e-07,1.678000e-07,7.107200e-07,4.661900e-08,4,110,36,3.768287e+16,9.871345e+16,2500050000,0.0,3,1,253,181,6,5,0,0
389,1.460000e-06,7.997000e-07,2.362000e-07,1.728000e-08,5,69,8,1.583189e+03,2.092630e+03,200,0.0,4,1,0,0,10,10,0,0
139,1.278000e+00,1.243000e+00,1.952000e+00,1.237000e+00,2,807,140,2.086710e+13,2.352928e+14,100000000,1.26,5,1,845,548,256,9,981,3


In [12]:
print(train_set.dtypes)

MaxPrice (Quarter 1)                                     float64
MaxPrice (Quarter 2)                                     float64
MaxPrice (Quarter 3)                                     float64
MaxPrice (Quarter 4)                                     float64
Blockchain                                                 int64
the number of Transactions                                object
Token concentration ratio per holder                      object
Total Variance                                           float64
Variance of holders with more than 1% tokens             float64
Token balance                                             object
first deposits                                            object
Blockchain Type                                            int64
class                                                      int64
Google results for project title (first day)               int64
Google results for project title (project duration/2)      int64
Google results for projec

In [13]:
#Transfer all data to numeric values

#TODO: reference from notes

cols_to_clean = [
    'the number of Transactions',
    'Token concentration ratio per holder',
    'Token balance',
    'first deposits',
    'Google results for project website (first day)',
    'Google results for project website (duration/2)'
]

data_sets = [train_set, test_set, val_set]

for i, df in enumerate(data_sets):
    for col in cols_to_clean:
        data_sets[i][col] = pd.to_numeric(
            df[col].astype(str)
                   .str.replace('\xa0', '', regex=False)
                   .str.replace(',', '', regex=False)
                   .str.strip(),
            errors='coerce'
        )

train_set, test_set, val_set = data_sets

# Verify
print(train_set[cols_to_clean].dtypes)
print(train_set[cols_to_clean].isnull().sum())
print(train_set[cols_to_clean].describe())

the number of Transactions                         float64
Token concentration ratio per holder                 int64
Token balance                                      float64
first deposits                                     float64
Google results for project website (first day)     float64
Google results for project website (duration/2)    float64
dtype: object
the number of Transactions                         0
Token concentration ratio per holder               0
Token balance                                      1
first deposits                                     1
Google results for project website (first day)     1
Google results for project website (duration/2)    1
dtype: int64
       the number of Transactions  Token concentration ratio per holder  \
count                6.500000e+02                          6.500000e+02   
mean                 1.089732e+06                          7.062838e+06   
std                  1.037284e+07                          1.793588e+08   
m

In [14]:
#Check for skew

skew = train_set.skew()

skew

MaxPrice (Quarter 1)                                     25.494971
MaxPrice (Quarter 2)                                     25.495098
MaxPrice (Quarter 3)                                     25.495098
MaxPrice (Quarter 4)                                     25.495084
Blockchain                                                0.080036
the number of Transactions                               19.951485
Token concentration ratio per holder                     25.495076
Total Variance                                           25.495098
Variance of holders with more than 1% tokens             25.494589
Token balance                                            25.475478
first deposits                                           25.475473
Blockchain Type                                           0.405838
class                                                    -0.402642
Google results for project title (first day)             23.899508
Google results for project title (project duration/2)    14.75